In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import joblib

In [13]:
df = pd.read_csv("../dataset/clean_house_data.csv")

df.head()

,area_type,location,total_sqft,bath,balcony,price,BHK,price_per_sqft
0,Super built-up Area,Electronic City Phase II,1056.0,2.0,1.0,39.07,2,3699.810606
1,Plot Area,Chikka Tirupathi,2600.0,5.0,3.0,120.00,4,4615.384615
2,Built-up Area,Uttarahalli,1440.0,2.0,3.0,62.00,3,4305.555556
3,Super built-up Area,Lingadheeranahalli,1521.0,3.0,1.0,95.00,3,6245.890861
4,Super built-up Area,Kothanur,1200.0,2.0,1.0,51.00,2,4250.000000


In [14]:
df.info()

df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12013 entries, 0 to 12012
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   area_type       12013 non-null  object 
 1   location        12013 non-null  object 
 2   total_sqft      12013 non-null  float64
 3   bath            12013 non-null  float64
 4   balcony         12013 non-null  float64
 5   price           12013 non-null  float64
 6   BHK             12013 non-null  int64  
 7   price_per_sqft  12013 non-null  float64
dtypes: float64(5), int64(1), object(2)
memory usage: 750.9+ KB


,total_sqft,bath,balcony,price,BHK,price_per_sqft
count,12013.000000,12013.000000,12013.000000,12013.000000,12013.000000,12013.000000
mean,1542.315982,2.511779,1.587613,105.003648,2.607259,6206.082347
std,1181.094228,1.006207,0.808867,134.205666,0.922985,3985.518807
min,300.000000,1.000000,0.000000,9.000000,1.000000,267.829813
25%,1107.000000,2.000000,1.000000,48.450000,2.000000,4199.363057
50%,1285.000000,2.000000,2.000000,68.000000,2.000000,5252.525253
75%,1660.000000,3.000000,2.000000,110.000000,3.000000,6823.529412
max,52272.000000,13.000000,3.000000,2912.000000,13.000000,176470.588235


In [15]:
X = df.drop("price", axis=1)
X = X.drop("price_per_sqft", axis=1)
y = df["price"]

categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols),
    ]
)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

Training Shape : (9610, 6)
Testing Shape : (2403, 6)


In [17]:
def evaluate_model(model_name, model):

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    prediction = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, prediction)
    rmse = np.sqrt(mean_squared_error(y_test, prediction))
    r2 = r2_score(y_test, prediction)

    return {
        "Model": model_name,
        "MAE": round(mae,2),
        "RMSE": round(rmse,2),
        "R2 Score": round(r2,4),
        "Pipeline": pipeline
    }

In [18]:
results = []

results.append(
    evaluate_model(
        "Linear Regression",
        LinearRegression()
    )
)

results.append(
    evaluate_model(
        "Decision Tree",
        DecisionTreeRegressor(random_state=42)
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )
    )
)

results.append(
    evaluate_model(
        "Gradient Boosting",
        GradientBoostingRegressor(random_state=42)
    )
)

In [19]:
result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    by="R2 Score",
    ascending=False
)

result_df

,Model,MAE,RMSE,R2 Score,Pipeline
3,Gradient Boosting,31.53,80.33,0.6447,"(ColumnTransformer(transformers=[('cat', OneHo..."
2,Random Forest,26.82,83.45,0.6166,"(ColumnTransformer(transformers=[('cat', OneHo..."
0,Linear Regression,38.37,95.79,0.4949,"(ColumnTransformer(transformers=[('cat', OneHo..."
1,Decision Tree,33.02,119.98,0.2075,"(ColumnTransformer(transformers=[('cat', OneHo..."


In [20]:
best_model = result_df.iloc[0]

print("Best Model :", best_model["Model"])
print("R² Score :", best_model["R2 Score"])

Best Model : Gradient Boosting
R² Score : 0.6447


In [21]:
best_pipeline = best_model["Pipeline"]

joblib.dump(best_pipeline, "../model/model.pkl")

print("Best model saved successfully!")

Best model saved successfully!


# Model Training Summary

### Models Trained

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

### Evaluation Metrics

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

### Conclusion

Four machine learning models were trained and evaluated. The model with the highest R² Score was selected as the final prediction model and saved for deployment.